In [2]:
import json
import pandas as pd
from datetime import datetime

The exported .json file from Wireshark a lot of times contains duplicate keys in an object, e.g. when there are multiple quic packets in one UDP packet.

In the .json we can see something like:

    "quic": {
    ...
    "quic.frame": { "quic.frame_type": "0x02", ... },
    "quic.frame": { "quic.frame_type": "0x18", ... },
    "quic.frame": { "quic.frame_type": "0x1e", ... },
    ...
    }

For this reason we need a custom JSON parser hook, that handles this problem, as the standard json.load() just overwrites the previous object in this case

In [21]:
def as_list_on_duplicate_keys(ordered_pairs):
    """
    A custom JSON object_pairs_hook that collects values for duplicate
    keys into a list.
    """
    d = {}
    for k, v in ordered_pairs:
        if k in d:
            if isinstance(d[k], list):
                d[k].append(v)
            else:
                d[k] = [d[k], v]
        else:
            d[k] = v
    return d


In [ ]:
def analyze_quic_capture(json_file_path):
    """
    Analyzes a decrypted QUIC packet capture from a Wireshark JSON export
    and extracts a set of high-level features for connection migration analysis.

    Args:
        json_file_path (str): The path to the JSON file.

    Returns:
        dict: A dictionary containing the extracted features for the flow.
              Returns None if the capture is empty or invalid.
    """
    with open(json_file_path, 'r') as f:
        packets = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)

    if not packets:
        print("Capture file is empty.")
        return None

    features = {}

    # --- 1. Initialization and Initial Packet Analysis ---
    first_packet = packets[0]['_source']['layers']
    last_packet = packets[-1]['_source']['layers']

    initial_ip_client = first_packet['ip']['ip.src']
    initial_ip_server = first_packet['ip']['ip.dst']
    initial_port_client = int(first_packet['udp']['udp.srcport'])
    initial_port_server = int(first_packet['udp']['udp.dstport'])
    
    time_first_epoch = float(first_packet['frame']['frame.time_epoch'])
    time_last_epoch = float(last_packet['frame']['frame.time_epoch'])

    features['ID'] = 1  # Static ID for this single flow analysis
    features['initial_ip_client'] = initial_ip_client
    features['initial_ip_server'] = initial_ip_server
    features['initial_port_client'] = initial_port_client
    features['initial_port_server'] = initial_port_server
    features['time_first'] = datetime.fromtimestamp(time_first_epoch).isoformat(sep='T', timespec='microseconds').replace(":", "-")
    features['time_last'] = datetime.fromtimestamp(time_last_epoch).isoformat(sep='T', timespec='microseconds').replace(":", "-")
    features['connection_duration'] = (time_last_epoch - time_first_epoch) * 1000  # in ms

    # Initialize counters and flags
    bytes_sent_client = 0
    bytes_sent_server = 0
    packets_sent_client = 0
    packets_sent_server = 0
    padding_bytes_in_validation_pc = 0
    padding_bytes_in_validation_pr = 0
    mtu = 0
    features['version_negotiation_occurred'] = 0
    features['retry_occurred'] = 0
    features['new_connection_ids_issued_server'] = 0
    features['retired_cid_count_client'] = 0
    features['retired_cid_count_server'] = 0
    features['padding_bytes_in_validation_pc'] = 0
    features['padding_bytes_in_validation_pr'] = 0
    features['mtu'] = 0

    features['new_connection_ids_issued_server'] = 0
    features['new_connection_ids_issued_client'] = 0
    

    
    # State variables
    migrated = False
    migrated_ip_client = None
    migrated_port_client = None
    time_to_migration = None
    packets_before_migration = 0
    
    # --- UPDATED: Handshake Duration variables ---
    handshake_start_time = time_first_epoch
    handshake_end_time = None # We will find the timestamp of the LAST 'Finished' message
    
    path_challenge_data = None
    path_challenge_time = None
    migration_start_time = None
    
    http_streams = set()
    app_data_before_migration = 0
    http_req_bytes = 0
    http_resp_bytes = 0

    # --- 2. Iterate Through All Packets ---
    for i, pkt_data in enumerate(packets):
        layers = pkt_data['_source']['layers']
        
        # Basic packet info
        current_time = float(layers['frame']['frame.time_epoch'])
        packet_len = int(layers['frame']['frame.len'])
        
        src_ip = layers['ip']['ip.src']
        dst_ip = layers['ip']['ip.dst']

        src_port = int(layers['udp']['udp.srcport'])
        dst_port = int(layers['udp']['udp.dstport'])
        
        if not migrated and \
           (dst_ip == initial_ip_server and dst_port == initial_port_server) and \
           (src_ip != initial_ip_client or src_port != initial_port_client):
            
            migrated = True
            migration_start_time = current_time
            migrated_ip_client = src_ip
            migrated_port_client = src_port
            
            time_to_migration = (current_time - time_first_epoch) * 1000
            packets_before_migration = i
            
            ip_changed = src_ip != initial_ip_client
            port_changed = src_port != initial_port_client
            if ip_changed and port_changed:
                features['migration_type'] = 'IP_AND_PORT'
            elif ip_changed:
                features['migration_type'] = 'IP_ONLY'
            elif port_changed:
                features['migration_type'] = 'PORT_ONLY'


        is_client_pkt = (src_ip == initial_ip_client or src_ip == migrated_ip_client)

        # Update byte and packet counters
        if is_client_pkt:
            bytes_sent_client += packet_len
            packets_sent_client += 1
        else:
            bytes_sent_server += packet_len
            packets_sent_server += 1

        print(f'frame:{i}, ' , layers)
        quic_packet_list = layers['quic']
        if not isinstance(quic_packet_list, list):
            quic_packet_list = [quic_packet_list]

        # 2. Loop through each QUIC packet within the UDP datagram.
        for quic_packet in quic_packet_list:
            if quic_packet.get('quic.version') == '0x00000000':
                features['version_negotiation_occurred'] = 1
            if quic_packet.get('quic.long.packet_type') == '3': # Retry packet
                features['retry_occurred'] = 1

            # Analyze QUIC frames if they exist
            # 3. Get frames from the current quic_packet, not from layers['quic'].
            quic_frames = quic_packet.get('quic.frame', [])
            if not isinstance(quic_frames, list): # If there's only one frame, it's a dict
                quic_frames = [quic_frames]
            
            # These flags help identify which packet contains padding
            is_path_challenge_pkt = any(f.get('quic.frame_type') == '0x000000000000001a' for f in quic_frames)
            is_path_response_pkt = any(f.get('quic.frame_type') == '0x000000000000001b' for f in quic_frames)
                
            for frame in quic_frames:
                frame_type = frame.get('quic.frame_type')
                if not frame_type: continue # Skip if frame is empty
                
                if frame_type == '0x0000000000000006': # CRYPTO frame
                    tls_handshake_msgs = frame.get('tls', {}).get('tls.handshake', [])
                    if not isinstance(tls_handshake_msgs, list):
                        tls_handshake_msgs = [tls_handshake_msgs]
                    
                    for msg in tls_handshake_msgs:
                        if msg and msg.get('tls.handshake.type') == '20':
                            handshake_end_time = current_time
                
                if frame_type == '0x0000000000000018': # NEW_CONNECTION_ID
                    if not is_client_pkt:
                        features['new_connection_ids_issued_server'] += 1
                    else:
                        # Note: clients do not issue CIDs, but keeping your logic
                        features['new_connection_ids_issued_client'] += 1
                
                if frame_type == '0x0000000000000019': # RETIRE_CONNECTION_ID
                    if is_client_pkt:
                        features['retired_cid_count_client'] += 1
                    else:
                        features['retired_cid_count_server'] += 1

                if frame_type == '0x000000000000001a' and is_client_pkt: # PATH_CHALLENGE
                    features['path_validation_initiated'] = 1
                    path_challenge_data = frame['quic.path_challenge.data']
                    path_challenge_time = current_time
                    mtu = max(mtu, packet_len)

                if frame_type == '0x000000000000001b' and not is_client_pkt: # PATH_RESPONSE
                    if path_challenge_data and frame['quic.path_response.data'] == path_challenge_data:
                        features['path_validation_response_latency'] = (current_time - path_challenge_time) * 1000
                        features['migration_duration'] = (current_time - migration_start_time) * 1000
                
                # --- IMPORTANT FIX for PADDING ---
                # Padding is its own frame type, not a field on other frames.
                if frame_type == '0x0000000000000000': # PADDING frame
                    padding_len = int(frame.get('quic.padding_length', 1)) # Padding can be 1 byte if length is omitted
                    if is_path_challenge_pkt:
                        padding_bytes_in_validation_pc = padding_len
                    if is_path_response_pkt:
                        padding_bytes_in_validation_pr = padding_len

                if frame_type in ('0x000000000000001c', '0x000000000000001d'): # CONNECTION_CLOSE
                    features['connection_close_type'] = 'CLIENT_CLOSE' if is_client_pkt else 'SERVER_CLOSE'    
            
        # HTTP/3 Analysis
        if 'http3' in layers:
            # Simple stream counting by checking for headers
            if 'http3.headers' in layers['http3'].get('http3.stream', {}):
                stream_id = layers['quic']['quic.frame']['quic.stream.stream_id']
                http_streams.add(stream_id)

            # Sum app data bytes from HTTP DATA frames
            http_frames = layers['http3'].get('http3.stream', {}).get('http3.frame', [])
            if not isinstance(http_frames, list):
                http_frames = [http_frames]
            
            for h3_frame in http_frames:
                if h3_frame.get('http3.frame_type') == '0x0000000000000000': # DATA frame
                    data_len = int(h3_frame.get('http3.frame_length', 0))
                    if not migrated:
                        app_data_before_migration += data_len
                    if is_client_pkt:
                        http_req_bytes += data_len
                    else:
                        http_resp_bytes += data_len


    # --- 3. Final Calculations and Assembly ---
    features['bytes_sent_client'] = bytes_sent_client
    features['bytes_sent_server'] = bytes_sent_server
    features['packets_sent_client'] = packets_sent_client
    features['packets_sent_server'] = packets_sent_server
    
    if handshake_end_time:
        features['handshake_duration'] = (handshake_end_time - handshake_start_time) * 1000
    else:
        features['handshake_duration'] = None # Handshake did not complete or was not found
        
    features['time_to_migration'] = time_to_migration
    features['packets_before_migration'] = packets_before_migration
    features['app_data_bytes_before_migration'] = app_data_before_migration
    
    # Set migration-related features to 0 or null if no migration occurred
    if not migrated:
        features['migration_type'] = 'NO_CHANGE'
        features['time_to_migration'] = 0
        features['packets_before_migration'] = 0
        features['migration_duration'] = 0
        features['path_validation_initiated'] = 0
        features['path_validation_response_latency'] = 0

    # These features couldn't be accurately determined from this specific JSON structure but are included as placeholders
    features['padding_bytes_in_validation_pc'] = padding_bytes_in_validation_pc
    features['padding_bytes_in_validation_pr'] = padding_bytes_in_validation_pr
    features['mtu'] = mtu
    
    features['total_http_streams'] = len(http_streams)
    features['http_request_response_byte_ratio'] = http_req_bytes / http_resp_bytes if http_resp_bytes > 0 else 0

    # Final check for connection close type
    if 'connection_close_type' not in features:
        features['connection_close_type'] = 'IDLE_TIMEOUT' # Default assumption
        
    return features

In [31]:
json_file = 'captures/migration_decrypted_export.json'
extracted_features = analyze_quic_capture(json_file)


frame:0,  {'frame': {'frame.section_number': '1', 'frame.interface_id': '0', 'frame.interface_id_tree': {'frame.interface_name': '\\Device\\NPF_Loopback', 'frame.interface_description': 'Adapter for loopback traffic capture'}, 'frame.encap_type': '15', 'frame.time': 'Sep 27, 2025 16:57:24.038823000 W. Europe Daylight Time', 'frame.time_utc': 'Sep 27, 2025 14:57:24.038823000 UTC', 'frame.time_epoch': '1758985044.038823000', 'frame.offset_shift': '0.000000000', 'frame.time_delta': '0.051386000', 'frame.time_delta_displayed': '0.000000000', 'frame.time_relative': '7.895238000', 'frame.number': '371', 'frame.len': '1232', 'frame.cap_len': '1232', 'frame.marked': '0', 'frame.ignored': '0', 'frame.protocols': 'null:ip:udp:quic', 'frame.coloring_rule.name': 'UDP', 'frame.coloring_rule.string': 'udp'}, 'null': {'null.family': '2'}, 'ip': {'ip.version': '4', 'ip.hdr_len': '20', 'ip.dsfield': '0x00', 'ip.dsfield_tree': {'ip.dsfield.dscp': '0', 'ip.dsfield.ecn': '0'}, 'ip.len': '1228', 'ip.id': '

In [13]:
pd.set_option('display.max_columns', None)

In [32]:
if extracted_features:
    df = pd.DataFrame([extracted_features])
    print("Extracted Features (Corrected Migration & Direction Logic):")
    display(df)

Extracted Features (Corrected Migration & Direction Logic):


,ID,initial_ip_client,initial_ip_server,initial_port_client,initial_port_server,time_first,time_last,connection_duration,version_negotiation_occurred,retry_occurred,new_connection_ids_issued_server,retired_cid_count_client,retired_cid_count_server,padding_bytes_in_validation_pc,padding_bytes_in_validation_pr,mtu,new_connection_ids_issued_client,migration_type,path_validation_initiated,path_validation_response_latency,migration_duration,connection_close_type,bytes_sent_client,bytes_sent_server,packets_sent_client,packets_sent_server,handshake_duration,time_to_migration,packets_before_migration,app_data_bytes_before_migration,total_http_streams,http_request_response_byte_ratio
0,1,127.0.0.3,127.0.0.1,49669,4433,2025-09-27T16-57-24.038823,2025-09-27T16-57-24.048266,9.443045,1,1,1,0,0,2597,1294,1382,1,IP_AND_PORT,1,0.463963,0.463963,CLIENT_CLOSE,7269,4568,14,12,5.47719,6.771088,13,0,0,0.0
